# 01_emissions_extraction.ipynb
**Author:** Emma McCallum  
**Purpose:** Extract and restructure BEGES emissions categories for five firms  
**Input:** data/processed/five_firms_full.csv  
**Output:** data/processed/emissions_long.csv

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/five_firms_full.csv")

print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

Rows: 7
Columns: 104


In [2]:
# Which firms and reporting years do we have?
df[["Raison sociale", "Année de reporting"]].sort_values("Raison sociale")

,Raison sociale,Année de reporting
0,ANTEA FRANCE,2019
1,ANTEA FRANCE,2023
2,ARTELIA,2022
3,BUREAU VERITAS EXPLOITATION,2019
4,BUREAU VERITAS EXPLOITATION,2023
5,ECOCERT SA,2024
6,Eco CO2,2023


## Note: multi-year data available for two firms

ANTEA FRANCE has reports for 2019 and 2023.  
BUREAU VERITAS EXPLOITATION has reports for 2019 and 2023.  

For the main cross-firm analysis we use the most recent report per firm.  
The 2019 reports are preserved in the raw data for a within-firm  
evolution analysis in a later notebook or as a bonus section.  

Potential question: did their Scope 3 coverage improve between 2019 and 2023?

In [3]:
# Keep only the most recent report per firm
df_recent = df.sort_values("Année de reporting", ascending=False)
df_recent = df_recent.drop_duplicates(subset="Raison sociale", keep="first")
df_recent = df_recent.sort_values("Raison sociale").reset_index(drop=True)

df_recent[["Raison sociale", "Année de reporting"]]

,Raison sociale,Année de reporting
0,ANTEA FRANCE,2023
1,ARTELIA,2022
2,BUREAU VERITAS EXPLOITATION,2023
3,ECOCERT SA,2024
4,Eco CO2,2023


In [4]:
# Find all emissions columns
emissions_cols = [col for col in df_recent.columns if col.startswith("Emissions publication")]
print(f"Number of emissions columns: {len(emissions_cols)}")
print("\nColumn names:")
for col in emissions_cols:
    print(col)

Number of emissions columns: 22

Column names:
Emissions publication P1.1
Emissions publication P1.2
Emissions publication P1.3
Emissions publication P1.4
Emissions publication P1.5
Emissions publication P2.1
Emissions publication P2.2
Emissions publication P3.1
Emissions publication P3.2
Emissions publication P3.3
Emissions publication P3.4
Emissions publication P3.5
Emissions publication P4.1
Emissions publication P4.2
Emissions publication P4.3
Emissions publication P4.4
Emissions publication P4.5
Emissions publication P5.1
Emissions publication P5.2
Emissions publication P5.3
Emissions publication P5.4
Emissions publication P6.1


# BEGES Emissions Category Structure and GHG Protocol Mapping

**Sources**
- ADEME: https://bilans-ges.ademe.fr/ressources/points-cles-methodologiques
- ADEME: *Méthode pour la réalisation des bilans d'émissions de GES*, Version 4, Octobre 2016
- GHG Protocol Corporate Standard, revised edition 2004, Chapter 4
- GHG Protocol Corporate Value Chain (Scope 3) Standard, 2011, Chapter 5 Table 5.3
- ISO TR 14069: *Guide d'application de la norme ISO 14064-1*

---

The BEGES framework defines **3 regulatory categories** and **23 posts**, in accordance
with ISO 14064-1. Categories 1 and 2 (posts 1–7) are mandatory. Category 3 (posts 8–23)
is recommended but not obligatory under the regulation. For publication and export, the
ADEME platform groups the 23 posts into **6 P-categories**. These are two separate
organisational schemes for the same underlying data.

The mapping between BEGES posts and GHG Protocol Scope 3 categories is approximate,
not identical. A full post-level cross-reference is built in notebook 02 using the ADEME
methodology document as the primary reference.

---

## BEGES regulatory categories and posts

**Category 1 — Émissions directes** | ≈ Scope 1 | MANDATORY | Posts 1–5

| Post | Name |
|------|------|
| 1 | Émissions directes des sources fixes de combustion |
| 2 | Émissions directes des sources mobiles à moteur thermique |
| 3 | Émissions directes des procédés hors énergie |
| 4 | Émissions directes fugitives |
| 5 | Émissions issues de la biomasse (sols et forêts) |

**Category 2 — Émissions indirectes associées à l'énergie** | ≈ Scope 2 | MANDATORY | Posts 6–7

| Post | Name |
|------|------|
| 6 | Émissions indirectes liées à la consommation d'électricité |
| 7 | Émissions indirectes liées à la consommation de vapeur, chaleur ou froid |

**Category 3 — Autres émissions indirectes** | ≈ Scope 3 | RECOMMENDED | Posts 8–23

| Post | Name |
|------|------|
| 8 | Émissions liées à l'énergie non incluses dans les catégories 1 et 2 |
| 9 | Achats de produits ou services |
| 10 | Immobilisations de biens |
| 11 | Déchets |
| 12 | Transport de marchandises amont |
| 13 | Déplacements professionnels |
| 14 | Actifs en leasing amont |
| 15 | Investissements |
| 16 | Transport des visiteurs et des clients |
| 17 | Transport de marchandises aval |
| 18 | Utilisation des produits vendus |
| 19 | Fin de vie des produits vendus |
| 20 | Franchise aval |
| 21 | Leasing aval |
| 22 | Déplacements domicile-travail |
| 23 | Autres émissions indirectes |

---

## ADEME export: P-category groupings

| P category | Regulatory category | Posts | GHG Protocol ≈ |
|------------|-------------------|-------|----------------|
| P1 — Émissions directes | Category 1 | 1–5 | Scope 1 |
| P2 — Émissions indirectes associées à l'énergie | Category 2 | 6–7 | Scope 2 |
| P3 — Émissions indirectes associées au transport | Category 3 | 12, 13, 16, 17, 22 | Scope 3 Cat 4, 6, 7, 9 |
| P4 — Émissions indirectes associées aux produits achetés | Category 3 | 8, 9, 10, 11, 14 | Scope 3 Cat 1, 2, 3, 5, 8 |
| P5 — Émissions indirectes associées aux produits vendus | Category 3 | 18, 19, 20, 21 | Scope 3 Cat 11, 12, 13, 14 |
| P6 — Autres émissions indirectes | Category 3 | 15, 23 | Scope 3 Cat 15 |

---

## GHG Protocol Scope 3: 15 categories

Official structure from *Corporate Value Chain (Scope 3) Standard*, Table 5.3.

**Upstream**

| Cat | Name |
|-----|------|
| 1 | Purchased goods and services |
| 2 | Capital goods |
| 3 | Fuel and energy related activities |
| 4 | Upstream transportation and distribution |
| 5 | Waste generated in operations |
| 6 | Business travel |
| 7 | Employee commuting |
| 8 | Upstream leased assets |

**Downstream**

| Cat | Name |
|-----|------|
| 9 | Downstream transportation and distribution |
| 10 | Processing of sold products |
| 11 | Use of sold products |
| 12 | End of life treatment of sold products |
| 13 | Downstream leased assets |
| 14 | Franchises |
| 15 | Investments |

**Note:** GHG Protocol Category 10 (Processing of sold products) has no direct BEGES
equivalent. It will appear as a structural gap in the notebook 02 coverage matrix.

---

In [7]:
# Extract all emissions columns for the five firms
emissions_cols = [col for col in df_recent.columns 
                  if col.startswith("Emissions publication")]

# Build a clean emissions dataframe with firm name and year
df_emissions = df_recent[["Raison sociale", "Année de reporting"] + emissions_cols].copy()

# Rename columns to shorter names for easier working
rename_map = {col: col.replace("Emissions publication ", "") 
              for col in emissions_cols}
df_emissions = df_emissions.rename(columns=rename_map)

print(df_emissions.shape)
df_emissions

(5, 24)


,Raison sociale,Année de reporting,P1.1,P1.2,P1.3,P1.4,P1.5,P2.1,P2.2,P3.1,...,P4.1,P4.2,P4.3,P4.4,P4.5,P5.1,P5.2,P5.3,P5.4,P6.1
0,ANTEA FRANCE,2023,104.30,1820.50,0.0,0.0,0.0,44.40,0.00,11.50,...,1283.30,1331.40,63.70,1315.5,8162.90,0.00,0.0,0.00,0.0,0.00
1,ARTELIA,2022,5.00,3088.00,0.0,272.0,0.0,649.00,332.00,0.00,...,1966.00,5047.00,38.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00
2,BUREAU VERITAS EXPLOITATION,2023,0.00,136.00,0.0,94.0,0.0,0.00,0.00,0.00,...,4912.00,395.00,62.00,0.0,16446.00,0.00,0.0,0.00,0.0,0.00
3,ECOCERT SA,2024,0.00,565.30,0.0,0.0,0.0,12.10,0.00,90.40,...,362.00,329.80,0.80,396.4,981.70,0.00,0.0,0.00,0.0,0.00
4,Eco CO2,2023,13.11,13.31,0.0,0.0,0.0,0.31,1.11,0.15,...,1353.04,78.63,0.23,0.0,3.48,0.45,0.0,0.63,0.0,21.64


## Wide to long format transformation

Currently the data is in wide format: one row per firm, one column per emissions category.
For analysis and visualisation we need long format: one row per firm per category.
This makes it easy to filter, group, and plot by category or by scope.
We use pandas melt() to do this transformation.

In [8]:
# Transform from wide to long format
df_long = df_emissions.melt(
    id_vars=["Raison sociale", "Année de reporting"],
    var_name="beges_category",
    value_name="tco2e"
)

print(f"Rows in long format: {len(df_long)}")
df_long.head(10)

Rows in long format: 110


,Raison sociale,Année de reporting,beges_category,tco2e
0,ANTEA FRANCE,2023,P1.1,104.30
1,ARTELIA,2022,P1.1,5.00
2,BUREAU VERITAS EXPLOITATION,2023,P1.1,0.00
3,ECOCERT SA,2024,P1.1,0.00
4,Eco CO2,2023,P1.1,13.11
5,ANTEA FRANCE,2023,P1.2,1820.50
6,ARTELIA,2022,P1.2,3088.00
7,BUREAU VERITAS EXPLOITATION,2023,P1.2,136.00
8,ECOCERT SA,2024,P1.2,565.30
9,Eco CO2,2023,P1.2,13.31


In [9]:
# Map each BEGES P-category to GHG Protocol scope equivalent
# Source: ADEME Méthode BEGES V4, 2016 and ADEME export documentation
scope_map = {
    "P1.1": "Scope 1", "P1.2": "Scope 1", "P1.3": "Scope 1",
    "P1.4": "Scope 1", "P1.5": "Scope 1",
    "P2.1": "Scope 2", "P2.2": "Scope 2",
    "P3.1": "Scope 3", "P3.2": "Scope 3", "P3.3": "Scope 3",
    "P3.4": "Scope 3", "P3.5": "Scope 3",
    "P4.1": "Scope 3", "P4.2": "Scope 3", "P4.3": "Scope 3",
    "P4.4": "Scope 3", "P4.5": "Scope 3",
    "P5.1": "Scope 3", "P5.2": "Scope 3", "P5.3": "Scope 3",
    "P5.4": "Scope 3",
    "P6.1": "Scope 3"
}

# Add mandatory vs recommended flag
mandatory_map = {
    "P1.1": "Mandatory", "P1.2": "Mandatory", "P1.3": "Mandatory",
    "P1.4": "Mandatory", "P1.5": "Mandatory",
    "P2.1": "Mandatory", "P2.2": "Mandatory",
    "P3.1": "Recommended", "P3.2": "Recommended", "P3.3": "Recommended",
    "P3.4": "Recommended", "P3.5": "Recommended",
    "P4.1": "Recommended", "P4.2": "Recommended", "P4.3": "Recommended",
    "P4.4": "Recommended", "P4.5": "Recommended",
    "P5.1": "Recommended", "P5.2": "Recommended", "P5.3": "Recommended",
    "P5.4": "Recommended",
    "P6.1": "Recommended"
}

df_long["scope"] = df_long["beges_category"].map(scope_map)
df_long["regulatory_status"] = df_long["beges_category"].map(mandatory_map)

print(df_long["scope"].value_counts())
df_long.head(10)

scope
Scope 3    75
Scope 1    25
Scope 2    10
Name: count, dtype: int64


,Raison sociale,Année de reporting,beges_category,tco2e,scope,regulatory_status
0,ANTEA FRANCE,2023,P1.1,104.30,Scope 1,Mandatory
1,ARTELIA,2022,P1.1,5.00,Scope 1,Mandatory
2,BUREAU VERITAS EXPLOITATION,2023,P1.1,0.00,Scope 1,Mandatory
3,ECOCERT SA,2024,P1.1,0.00,Scope 1,Mandatory
4,Eco CO2,2023,P1.1,13.11,Scope 1,Mandatory
5,ANTEA FRANCE,2023,P1.2,1820.50,Scope 1,Mandatory
6,ARTELIA,2022,P1.2,3088.00,Scope 1,Mandatory
7,BUREAU VERITAS EXPLOITATION,2023,P1.2,136.00,Scope 1,Mandatory
8,ECOCERT SA,2024,P1.2,565.30,Scope 1,Mandatory
9,Eco CO2,2023,P1.2,13.31,Scope 1,Mandatory


In [10]:
# Total emissions per firm per scope
scope_totals = df_long.groupby(
    ["Raison sociale", "Année de reporting", "scope"]
)["tco2e"].sum().reset_index()

scope_totals = scope_totals.rename(columns={"tco2e": "total_tco2e"})
scope_totals["total_tco2e"] = scope_totals["total_tco2e"].round(1)

scope_totals

,Raison sociale,Année de reporting,scope,total_tco2e
0,ANTEA FRANCE,2023,Scope 1,1924.8
1,ANTEA FRANCE,2023,Scope 2,44.4
2,ANTEA FRANCE,2023,Scope 3,12840.8
3,ARTELIA,2022,Scope 1,3365.0
4,ARTELIA,2022,Scope 2,981.0
5,ARTELIA,2022,Scope 3,17944.0
6,BUREAU VERITAS EXPLOITATION,2023,Scope 1,230.0
7,BUREAU VERITAS EXPLOITATION,2023,Scope 2,0.0
8,BUREAU VERITAS EXPLOITATION,2023,Scope 3,23068.0
9,ECOCERT SA,2024,Scope 1,565.3


In [11]:
# Calculate Scope 3 as percentage of total reported emissions
firm_totals = scope_totals.groupby("Raison sociale")["total_tco2e"].sum().reset_index()
firm_totals = firm_totals.rename(columns={"total_tco2e": "grand_total"})

scope3_totals = scope_totals[scope_totals["scope"] == "Scope 3"][
    ["Raison sociale", "total_tco2e"]
].rename(columns={"total_tco2e": "scope3_total"})

firm_summary = firm_totals.merge(scope3_totals, on="Raison sociale")
firm_summary["scope3_pct"] = (
    firm_summary["scope3_total"] / firm_summary["grand_total"] * 100
).round(1)

firm_summary = firm_summary.sort_values("scope3_pct", ascending=False)
firm_summary

,Raison sociale,grand_total,scope3_total,scope3_pct
2,BUREAU VERITAS EXPLOITATION,23298.0,23068.0,99.0
4,Eco CO2,1563.4,1535.6,98.2
0,ANTEA FRANCE,14810.0,12840.8,86.7
3,ECOCERT SA,3761.0,3183.6,84.6
1,ARTELIA,22290.0,17944.0,80.5


In [12]:
# Add headcount for emissions intensity calculation
headcount_col = "Nombre de salariés/d'agents de l'ensemble des SIREN déclarés sur ce bilan lors de l'année de reporting du bilan"

df_headcount = df_recent[["Raison sociale", headcount_col]].copy()
df_headcount = df_headcount.rename(columns={headcount_col: "headcount"})
df_headcount["headcount"] = pd.to_numeric(df_headcount["headcount"], errors="coerce")

firm_summary = firm_summary.merge(df_headcount, on="Raison sociale")
firm_summary["tco2e_per_employee"] = (
    firm_summary["scope3_total"] / firm_summary["headcount"]
).round(1)

firm_summary

,Raison sociale,grand_total,scope3_total,scope3_pct,headcount,tco2e_per_employee
0,BUREAU VERITAS EXPLOITATION,23298.0,23068.0,99.0,NaN,NaN
1,Eco CO2,1563.4,1535.6,98.2,NaN,NaN
2,ANTEA FRANCE,14810.0,12840.8,86.7,NaN,NaN
3,ECOCERT SA,3761.0,3183.6,84.6,778.0,4.1
4,ARTELIA,22290.0,17944.0,80.5,NaN,NaN


## Note on headcount data

Headcount was missing from the ADEME export for four of the five firms.
ECOCERT SA is the only firm with headcount data available in the dataset:
778 employees, 2024, giving 4.1 tCO2e Scope 3 per employee.

For the remaining four firms, headcount data was not available in the ADEME export.
Emissions intensity per employee cannot be calculated for those firms without
sourcing headcount from external sources such as company annual reports.
This is flagged as a data quality limitation. Notebook 03 will revisit this
when reviewing the methodology fields of each firm's published BEGES report.

In [13]:
# Save the long format emissions dataframe
df_long.to_csv("../data/processed/emissions_long.csv", index=False)

# Save the firm summary
firm_summary.to_csv("../data/processed/firm_summary.csv", index=False)

print("Saved emissions_long.csv and firm_summary.csv")
print(f"emissions_long: {len(df_long)} rows")
print(f"firm_summary: {len(firm_summary)} rows")

Saved emissions_long.csv and firm_summary.csv
emissions_long: 110 rows
firm_summary: 5 rows


In [14]:
df_long[df_long["beges_category"] == "P6.1"][
    ["Raison sociale", "tco2e"]
]

,Raison sociale,tco2e
105,ANTEA FRANCE,0.00
106,ARTELIA,0.00
107,BUREAU VERITAS EXPLOITATION,0.00
108,ECOCERT SA,0.00
109,Eco CO2,21.64


## Important: treatment of BEGES P6

P6 (autres émissions indirectes) in the BEGES framework refers to avoided emissions,
not additional indirect emissions. Avoided emissions represent emissions that did not
occur because of an action the organisation took, for example renewable energy generation
displacing grid electricity. They are a separate accounting concept and cannot be
included in a Scope 3-equivalent total.

Under both BEGES and GHG Protocol logic, avoided emissions must be reported separately
and never netted against gross emissions totals.

Eco CO2 is the only firm in this dataset with a non-zero P6.1 value (21.64 tCO2e).
This will be investigated in notebook 03 to determine whether this represents
genuine avoided emissions or a misclassification.

Scope 3-equivalent totals in this notebook are calculated from P3, P4, and P5 only.
P6 is excluded and reported separately below.

In [15]:
# Separate P6 avoided emissions from Scope 3-equivalent totals
df_p6 = df_long[df_long["beges_category"] == "P6.1"][
    ["Raison sociale", "Année de reporting", "tco2e"]
].rename(columns={"tco2e": "p6_avoided_emissions"})

print("P6 avoided emissions by firm:")
print(df_p6.to_string(index=False))

# Recalculate Scope 3-equivalent excluding P6
df_scope3_clean = df_long[
    (df_long["scope"] == "Scope 3") & 
    (df_long["beges_category"] != "P6.1")
]

scope3_clean_totals = df_scope3_clean.groupby(
    "Raison sociale"
)["tco2e"].sum().reset_index()
scope3_clean_totals.columns = ["Raison sociale", "scope3_equivalent_tco2e"]
scope3_clean_totals["scope3_equivalent_tco2e"] = scope3_clean_totals[
    "scope3_equivalent_tco2e"
].round(2)

print("\nScope 3-equivalent totals (P3+P4+P5 only, P6 excluded):")
print(scope3_clean_totals.to_string(index=False))

P6 avoided emissions by firm:
             Raison sociale  Année de reporting  p6_avoided_emissions
               ANTEA FRANCE                2023                  0.00
                    ARTELIA                2022                  0.00
BUREAU VERITAS EXPLOITATION                2023                  0.00
                 ECOCERT SA                2024                  0.00
                    Eco CO2                2023                 21.64

Scope 3-equivalent totals (P3+P4+P5 only, P6 excluded):
             Raison sociale  scope3_equivalent_tco2e
               ANTEA FRANCE                 12840.80
                    ARTELIA                 17944.00
BUREAU VERITAS EXPLOITATION                 23068.00
                 ECOCERT SA                  3183.60
                    Eco CO2                  1513.92


In [16]:
# Update firm_summary with corrected Scope 3-equivalent totals
firm_summary_clean = firm_summary.merge(
    scope3_clean_totals, on="Raison sociale"
)

# Recalculate Scope 3 percentage using corrected totals
firm_summary_clean["scope3_pct_clean"] = (
    firm_summary_clean["scope3_equivalent_tco2e"] /
    firm_summary_clean["grand_total"] * 100
).round(1)

firm_summary_clean[[
    "Raison sociale",
    "grand_total",
    "scope3_equivalent_tco2e",
    "scope3_pct_clean"
]].sort_values("scope3_pct_clean", ascending=False)

,Raison sociale,grand_total,scope3_equivalent_tco2e,scope3_pct_clean
0,BUREAU VERITAS EXPLOITATION,23298.0,23068.00,99.0
1,Eco CO2,1563.4,1513.92,96.8
2,ANTEA FRANCE,14810.0,12840.80,86.7
3,ECOCERT SA,3761.0,3183.60,84.6
4,ARTELIA,22290.0,17944.00,80.5
